# Diabetes Classification using Feedforward Neural Networks

## 📚 Learning Objectives

By completing this notebook, you will:
- Build a feedforward classifier for a tabular dataset
- Preprocess data, train, and evaluate the model

## 🔗 Prerequisites

- ✅ Basic Python
- ✅ Basic NumPy/Pandas (when applicable)

---

---

# Diabetes Classification using Feedforward Neural Networks

**Unit:** Unit 5: Introduction to Generative AI and Course Summary  

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand binary classification problems
- Learn metrics: Accuracy, Precision, Recall, F1-score
- Perform EDA and data preprocessing for medical datasets
- Build and train FFNN for diabetes classification
- Handle missing data and imbalanced classes

---

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
print("=== Binary Classification Metrics ===")
print("\nMetrics for evaluating binary classification:")
print(" - Accuracy: (TP + TN) / (TP + TN + FP + FN)")
print(" - Precision: TP / (TP + FP) - How many predicted positives are actually positive?")
print(" - Recall: TP / (TP + FN) - How many actual positives did we catch?")
print(" - F1-score: 2 * (Precision * Recall) / (Precision + Recall) - Harmonic mean")


=== Binary Classification Metrics ===

Metrics for evaluating binary classification:
 - Accuracy: (TP + TN) / (TP + TN + FP + FN)
 - Precision: TP / (TP + FP) - How many predicted positives are actually positive?
 - Recall: TP / (TP + FN) - How many actual positives did we catch?
 - F1-score: 2 * (Precision * Recall) / (Precision + Recall) - Harmonic mean


## Example: Diabetes Classification with FFNN


In [2]:
import pandas as pd
import numpy as np
# Create sample medical dataset (simplified)
np.random.seed(42)
n_samples = 1000

# Features: age, BMI, glucose, blood pressure, etc.
data = {
 'age': np.random.randint(20, 80, n_samples),
 'bmi': np.random.normal(25, 5, n_samples),
 'glucose': np.random.normal(120, 30, n_samples),
 'blood_pressure': np.random.normal(80, 10, n_samples),
 'insulin': np.random.normal(100, 40, n_samples)
}

df = pd.DataFrame(data)

# Create target: diabetes (1) or no diabetes (0)
# Higher glucose and BMI increase diabetes risk
diabetes_prob = 1 / (1 + np.exp(-(df['glucose'] - 120) / 30 - (df['bmi'] - 25) / 5))
df['diabetes'] = (np.random.random(n_samples) < diabetes_prob).astype(int)

print("=== Dataset Overview ===")
print(f"Total samples: {len(df)}")
print(f"Diabetes cases: {df['diabetes'].sum()} ({df['diabetes'].mean():.1%})")
print(f"No diabetes: {(1-df['diabetes']).sum()} ({(1-df['diabetes']).mean():.1%})")
print("\nFirst few rows:")
print(df.head())
# EDA: Check for missing values
print("\n=== EDA: Missing Values ===")
print(df.isnull().sum())

# Prepare data
X = df[['age', 'bmi', 'glucose', 'blood_pressure', 'insulin']].values
y = df['diabetes'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")


=== Dataset Overview ===
Total samples: 1000
Diabetes cases: 533 (53.3%)
No diabetes: 467 (46.7%)

First few rows:
   age        bmi     glucose  blood_pressure     insulin  diabetes
0   58  24.991820   99.492186       74.965789  122.493762         0
1   71  22.869838  141.286007       78.583902   84.487996         0
2   48  31.377316  113.659917       66.175401   88.088104         0
3   34  24.542170  116.477974       95.407858   52.931859         1
4   62  29.685363  119.185247       80.695481   99.402529         1

=== EDA: Missing Values ===
age               0
bmi               0
glucose           0
blood_pressure    0
insulin           0
diabetes          0
dtype: int64

Training set: 800 samples
Test set: 200 samples


In [3]:
# Build FFNN model
model = Sequential([
 Dense(64, activation='relu', input_shape=(5,)),
 Dense(32, activation='relu'),
 Dense(1, activation='sigmoid') # Binary classification
])

model.compile(
 optimizer='adam', loss='binary_crossentropy',
 metrics=['accuracy']
)

# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train model
history = model.fit(
 X_train_scaled, y_train,
 validation_split=0.2,
 epochs=100,
 batch_size=32,
 callbacks=[early_stopping],
 verbose=0
)

# Evaluate
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=== Model Performance ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Diabetes', 'Diabetes']))

/Users/abdullah/venvs/ai-diploma-tf/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 


=== Model Performance ===
Accuracy: 0.7350
Precision: 0.7700
Recall: 0.7196
F1-score: 0.7440

Classification Report:
              precision    recall  f1-score   support

 No Diabetes       0.70      0.75      0.73        93
    Diabetes       0.77      0.72      0.74       107

    accuracy                           0.73       200
   macro avg       0.73      0.74      0.73       200
weighted avg       0.74      0.73      0.74       200

